# Synthetic attribution experiment

Что заложено:
- 1000 synthetic users;
- 25% пользователей без рекламной метки;
- ground truth хранится отдельно и не попадает в таблицы, которые читает модель.

Ноутбук создаёт data/mock/synthetic_experiment.db. Сам файл .db не нужно коммитить.

Важно: расчёт атрибуции и ROMI здесь не переписан заново. Для расчётов используется текущий код из src/core/attribution.py.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from core.db import connect
from core import attribution

SEED = 42
N_USERS = 1000
WINDOW_DAYS = 30
ACQUIRING_RATE = 0.03

DB_PATH = ROOT / "data" / "mock" / "synthetic_experiment.db"
CHART_PATH = ROOT / "outputs" / "charts" / "synthetic_model_accuracy.png"

DB_PATH.parent.mkdir(parents=True, exist_ok=True)
CHART_PATH.parent.mkdir(parents=True, exist_ok=True)

placements = pd.DataFrame([
    {"placement_id": "a", "channel": "tg_ml_jobs", "campaign_name": "autumn_2026",
     "placement_type": "external", "creative": "creative_a", "post_category": "sale",
     "discount_value": 0, "cost": 50000, "true_effect": 0.22},
    {"placement_id": "b", "channel": "tg_data_jobs", "campaign_name": "autumn_2026",
     "placement_type": "external", "creative": "creative_b", "post_category": "native",
     "discount_value": 0, "cost": 35000, "true_effect": 0.16},
    {"placement_id": "c", "channel": "tg_students", "campaign_name": "autumn_2026",
     "placement_type": "external", "creative": "creative_c", "post_category": "discount",
     "discount_value": 15, "cost": 20000, "true_effect": 0.10},
    {"placement_id": "d", "channel": "tg_career", "campaign_name": "autumn_2026",
     "placement_type": "external", "creative": "creative_d", "post_category": "content",
     "discount_value": 0, "cost": 10000, "true_effect": 0.05},
])

# 25% пользователей вообще без рекламной метки.
SCENARIOS = {
    "organic_only": 0.25,
    "single_ad": 0.25,
    "two_ads": 0.20,
    "ad_then_organic": 0.10,
    "organic_then_ad": 0.10,
    "expired_ad": 0.05,
    "repeat_purchase": 0.05,
}

SURVEY_SOURCES = ["own", "ad", "friend", "search", "other", "skip"]

SINGLE_COURSES = [
    ("ML старт", 8950), ("ML про", 8950), ("AI агенты", 8950),
    ("Data Science", 6950), ("Аналитика старт", 8950),
    ("Алгоритмы старт", 8950), ("Backend старт", 8950),
    ("Линейная алгебра", 9950), ("К ВУЗу", 6950),
]

BUNDLES = [
    (["AI агенты", "ML про"], 14950),
    (["Аналитика старт", "Аналитика про"], 14950),
    (["ML старт", "Алгоритмы старт"], 14950),
    (["Алгоритмы старт", "Алгоритмы про"], 14950),
    (["ML старт", "ML про"], 14950),
]


In [ ]:
def make_scenario_pool(rng, n_users):
    names = list(SCENARIOS)
    counts = {name: int(round(share * n_users)) for name, share in SCENARIOS.items()}
    counts[names[0]] += n_users - sum(counts.values())

    pool = []
    for name, count in counts.items():
        pool.extend([name] * count)

    rng.shuffle(pool)
    return pool


def seasonality_bonus(ts):
    ts = pd.Timestamp(ts)
    if pd.Timestamp("2026-08-20") <= ts <= pd.Timestamp("2026-08-31"):
        return 0.10
    if pd.Timestamp("2026-09-01") <= ts <= pd.Timestamp("2026-09-10"):
        return 0.07
    return 0.02


def choose_product(rng):
    if rng.random() < 0.24:
        courses, amount = BUNDLES[rng.integers(len(BUNDLES))]
        return list(courses), float(amount)

    course, amount = SINGLE_COURSES[rng.integers(len(SINGLE_COURSES))]
    return [course], float(amount)


def effect_for(pid):
    return float(
        placements.loc[placements.placement_id.eq(pid), "true_effect"].iloc[0]
    )


def eligible_ads_for_ground_truth(user_touches, purchase_ts, previous_order_ts=None):
    """Только для генерации ground truth, не для расчёта атрибуции."""
    result = []

    for touch in user_touches:
        pid = touch["placement_id"]
        if pid is None:
            continue

        touch_ts = pd.Timestamp(touch["timestamp"])
        delta = pd.Timestamp(purchase_ts) - touch_ts

        if pd.Timedelta(0) <= delta <= pd.Timedelta(days=WINDOW_DAYS):
            if previous_order_ts is None or touch_ts > pd.Timestamp(previous_order_ts):
                result.append(pid)

    return result


def purchase_probability(ad_ids, purchase_ts):
    base = 0.10 + seasonality_bonus(purchase_ts)
    lift = sum(effect_for(pid) for pid in ad_ids)
    return min(0.85, base + lift)


def choose_true_driver(rng, ad_ids, purchase_ts):
    """Скрытый ground truth: модель его не получает."""
    weights = {None: 0.10 + seasonality_bonus(purchase_ts)}

    for pid in ad_ids:
        weights[pid] = weights.get(pid, 0) + effect_for(pid)

    keys = list(weights)
    probs = np.array([weights[k] for k in keys], dtype=float)
    probs /= probs.sum()

    return rng.choice(keys, p=probs)


In [ ]:
def generate_synthetic(seed=SEED, n_users=N_USERS):
    rng = np.random.default_rng(seed)
    scenario_pool = make_scenario_pool(rng, n_users)
    placement_ids = placements.placement_id.tolist()

    users, touches, events, leads, orders, order_items = [], [], [], [], [], []
    order_truth, user_truth = [], []

    touch_id = 1
    event_id = 1
    lead_id = 1
    item_id = 1

    start = pd.Timestamp("2026-08-01")
    end = pd.Timestamp("2026-09-05")
    span_days = (end - start).days

    for i in range(n_users):
        uid = f"u{i:04d}"
        scenario = scenario_pool[i]
        user_truth.append({"user_id_hash": uid, "scenario": scenario})

        t0 = start + pd.Timedelta(
            days=int(rng.integers(span_days + 1)),
            hours=int(rng.integers(8, 22)),
        )

        user_touches = []

        def add_touch(ts, pid):
            nonlocal touch_id

            row = {
                "touch_id": touch_id,
                "user_id_hash": uid,
                "placement_id": pid,
                "touch_type": "bot_start",
                "timestamp": pd.Timestamp(ts),
                "is_synthetic": 1,
            }

            touches.append(row)
            user_touches.append(row)
            touch_id += 1
            return row

        if scenario == "organic_only":
            add_touch(t0, None)

            if rng.random() < 0.35:
                add_touch(t0 + pd.Timedelta(days=int(rng.integers(1, 8))), None)

        elif scenario == "single_ad":
            add_touch(t0, rng.choice(placement_ids))

        elif scenario == "two_ads":
            p1, p2 = rng.choice(placement_ids, size=2, replace=False)
            add_touch(t0, p1)
            add_touch(t0 + pd.Timedelta(days=int(rng.integers(1, 8))), p2)

        elif scenario == "ad_then_organic":
            add_touch(t0, rng.choice(placement_ids))
            add_touch(t0 + pd.Timedelta(days=int(rng.integers(1, 6))), None)

        elif scenario == "organic_then_ad":
            add_touch(t0, None)
            add_touch(
                t0 + pd.Timedelta(days=int(rng.integers(1, 6))),
                rng.choice(placement_ids),
            )

        elif scenario == "expired_ad":
            add_touch(t0, rng.choice(placement_ids))

        elif scenario == "repeat_purchase":
            p1, _ = rng.choice(placement_ids, size=2, replace=False)
            add_touch(t0, p1)

        users.append({
            "user_id_hash": uid,
            "username_hash": None,
            "first_seen": t0,
            "first_placement_id": user_touches[0]["placement_id"],
            "source_system": "bot",
        })

        if rng.random() < 0.75:
            courses, _ = choose_product(rng)
            last_touch = user_touches[-1]

            events.append({
                "event_id": event_id,
                "user_id_hash": uid,
                "touch_id": last_touch["touch_id"],
                "event_type": "view_course",
                "item": courses[0],
                "timestamp": last_touch["timestamp"] + pd.Timedelta(minutes=5),
                "is_synthetic": 1,
            })

            event_id += 1

        if scenario == "expired_ad":
            # Специально создаём покупку позже 30 дней, чтобы проверить фильтр.
            purchase_times = [
                user_touches[-1]["timestamp"] + pd.Timedelta(days=int(rng.integers(31, 40)))
            ]

        elif scenario == "repeat_purchase":
            first_purchase_ts = (
                user_touches[-1]["timestamp"] + pd.Timedelta(days=int(rng.integers(1, 7)))
            )

            purchase_times = [first_purchase_ts]

            second_pid = rng.choice(
                [p for p in placement_ids if p != user_touches[-1]["placement_id"]]
            )

            add_touch(
                first_purchase_ts + pd.Timedelta(days=int(rng.integers(1, 5))),
                second_pid,
            )

            purchase_times.append(
                user_touches[-1]["timestamp"] + pd.Timedelta(days=int(rng.integers(1, 7)))
            )

        else:
            purchase_times = [
                user_touches[-1]["timestamp"] + pd.Timedelta(days=int(rng.integers(1, 11)))
            ]

        previous_order_ts = None
        made_order = False

        for order_no, purchase_ts in enumerate(purchase_times, start=1):
            ad_ids = eligible_ads_for_ground_truth(
                user_touches,
                purchase_ts,
                previous_order_ts,
            )

            # Эти два сценария нужны как обязательные тестовые примеры.
            if scenario in {"expired_ad", "repeat_purchase"}:
                should_buy = True
            else:
                should_buy = rng.random() < purchase_probability(ad_ids, purchase_ts)

            if not should_buy:
                continue

            made_order = True
            true_driver = choose_true_driver(rng, ad_ids, purchase_ts)
            courses, amount = choose_product(rng)

            last_before_purchase = max(
                [t for t in user_touches if t["timestamp"] <= purchase_ts],
                key=lambda x: x["timestamp"],
            )

            self_source = (
                rng.choice(SURVEY_SOURCES)
                if last_before_purchase["placement_id"] is None
                else None
            )

            leads.append({
                "lead_id": lead_id,
                "user_id_hash": uid,
                "touch_id": last_before_purchase["touch_id"],
                "course_interest": " + ".join(courses),
                "self_reported_source": self_source,
                "conversation_started_at": purchase_ts - pd.Timedelta(hours=2),
                "manager_id": "synthetic_manager",
                "status": "paid",
                "is_synthetic": 1,
            })

            order_id = f"o_{uid}_{order_no}"

            orders.append({
                "order_id": order_id,
                "user_id_hash": uid,
                "lead_id": lead_id,
                "student_id": None,
                "amount": amount,
                "variable_costs": round(amount * ACQUIRING_RATE, 2),
                "variable_costs_source": "rate_estimate",
                "timestamp": purchase_ts,
                "source_system": "bot",
                "is_synthetic": 1,
            })

            share = round(amount / len(courses), 2)

            for course in courses:
                order_items.append({
                    "item_id": item_id,
                    "order_id": order_id,
                    "course": course,
                    "amount": share,
                })
                item_id += 1

            order_truth.append({
                "order_id": order_id,
                "user_id_hash": uid,
                "true_driver": true_driver,
                "amount": amount,
                "timestamp": purchase_ts,
                "scenario": scenario,
            })

            lead_id += 1
            previous_order_ts = purchase_ts

        if not made_order and rng.random() < 0.20:
            last_touch = user_touches[-1]
            courses, _ = choose_product(rng)

            self_source = (
                rng.choice(SURVEY_SOURCES)
                if last_touch["placement_id"] is None
                else None
            )

            leads.append({
                "lead_id": lead_id,
                "user_id_hash": uid,
                "touch_id": last_touch["touch_id"],
                "course_interest": " + ".join(courses),
                "self_reported_source": self_source,
                "conversation_started_at": last_touch["timestamp"] + pd.Timedelta(hours=1),
                "manager_id": None,
                "status": "open",
                "is_synthetic": 1,
            })

            lead_id += 1

    placement_db = placements.drop(columns="true_effect").copy()
    placement_db["publication_time"] = pd.Timestamp("2026-08-01")
    placement_db["created_at"] = pd.Timestamp("2026-08-01")
    placement_db["is_synthetic"] = 1

    return {
        "dim_placement": placement_db,
        "dim_user": pd.DataFrame(users),
        "fact_touch": pd.DataFrame(touches),
        "fact_bot_event": pd.DataFrame(events),
        "fact_lead": pd.DataFrame(leads),
        "fact_order": pd.DataFrame(orders),
        "fact_order_item": pd.DataFrame(order_items),
        "order_truth": pd.DataFrame(order_truth),
        "user_truth": pd.DataFrame(user_truth),
    }


In [ ]:
def write_db(frames, path=DB_PATH):
    if path.exists():
        path.unlink()

    # Используем ту же схему SQLite, что и бот.
    conn = connect(str(path))

    for table in [
        "dim_placement",
        "dim_user",
        "fact_touch",
        "fact_bot_event",
        "fact_lead",
        "fact_order",
        "fact_order_item",
    ]:
        df = frames[table].copy()

        for col in df.columns:
            if pd.api.types.is_datetime64_any_dtype(df[col]):
                df[col] = df[col].dt.strftime("%Y-%m-%d %H:%M:%S")

        df.to_sql(table, conn, index=False, if_exists="append")

    conn.commit()
    return conn


frames = generate_synthetic()
conn = write_db(frames)

print("users:", len(frames["dim_user"]))
print("touches:", len(frames["fact_touch"]))
print("orders:", len(frames["fact_order"]))
print("db:", DB_PATH)

display(
    frames["user_truth"]["scenario"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("share_%")
    .to_frame()
)


## Проверка окна 30 дней

Фильтр уже реализован в src/core/attribution.py. Здесь мы его не дублируем: запускаем тот же расчёт и проверяем результат.

В synthetic-данных специально есть сценарий expired_ad: рекламное касание происходит за 31–39 дней до покупки. При окне 30 дней такая реклама не должна получить атрибуцию.


In [ ]:
# Запускаем ровно тот же код, который использует бот.
attribution.rebuild(conn, window_days=WINDOW_DAYS)

window_check = pd.read_sql_query(
    """
    SELECT
        a.attribution_model,
        a.order_id,
        a.placement_id,
        julianday(o.timestamp) - julianday(t.timestamp) AS days_before_order
    FROM fact_attribution a
    JOIN fact_order o ON o.order_id = a.order_id
    JOIN fact_touch t ON t.touch_id = a.touch_id
    WHERE a.touch_id IS NOT NULL
    """,
    conn,
)

assert window_check["days_before_order"].between(0, WINDOW_DAYS).all()

print(
    "Максимальный возраст рекламного касания, которое реально попало в атрибуцию:",
    round(window_check["days_before_order"].max(), 2),
    "дней",
)

expired_orders = frames["order_truth"].loc[
    frames["order_truth"]["scenario"].eq("expired_ad"),
    "order_id",
].tolist()

placeholders = ",".join("?" for _ in expired_orders)

expired_result = pd.read_sql_query(
    f"""
    SELECT order_id, placement_id, attribution_model
    FROM fact_attribution
    WHERE order_id IN ({placeholders})
      AND attribution_model = 'last'
    """,
    conn,
    params=expired_orders,
)

assert expired_result["placement_id"].isna().all()

print(
    "expired_ad: все покупки с рекламой старше 30 дней ушли в Organic/Direct — OK"
)


## Расчёт моделей и ROMI

Ниже нет собственной реализации формул атрибуции. Используются функции report, compare_models и compare_windows из src/core/attribution.py.


In [ ]:
model_comparison = attribution.compare_models(
    conn,
    window_days=WINDOW_DAYS,
)

print("ROMI по тем же моделям, которые использует бот:")
display(model_comparison)

print("Проверка чувствительности last touch к окну:")
display(
    attribution.compare_windows(
        conn,
        model="last",
        windows=(7, 30),
    )
)

print("Сводка last touch, окно 30 дней:")
last_summary = attribution.summary(
    conn,
    model="last",
    window_days=WINDOW_DAYS,
)

display(last_summary["placements"])


## Анализ относительно ground truth

Ground truth нужен только для synthetic-эксперимента. Он не записывается в рабочие таблицы и не участвует в расчёте бота.

Мы сравниваем заранее известный результат генератора с ROMI, который вернул существующий attribution.py.


In [ ]:
def true_romi(order_truth):
    true_revenue = (
        order_truth[order_truth["true_driver"].notna()]
        .groupby("true_driver")["amount"]
        .sum()
        .rename("true_revenue")
    )

    out = (
        placements[["placement_id", "cost"]]
        .set_index("placement_id")
        .join(true_revenue)
    )

    out["true_revenue"] = out["true_revenue"].fillna(0)
    out["true_romi"] = (
        out["true_revenue"] - out["cost"]
    ) / out["cost"]

    return out["true_romi"]


truth = true_romi(frames["order_truth"])

comparison = pd.DataFrame({"true_romi": truth}).join(model_comparison)

display(
    comparison.sort_values(
        "true_romi",
        ascending=False,
    )
)


In [ ]:
def evaluate_run(conn, order_truth):
    truth = true_romi(order_truth)
    model_romi = attribution.compare_models(
        conn,
        window_days=WINDOW_DAYS,
    )

    true_best = truth.idxmax()
    true_rank = truth.rank(ascending=False, method="average")

    rows = []

    for model in attribution.MODELS:
        predicted = model_romi[model].reindex(truth.index)
        predicted_best = predicted.idxmax()
        predicted_rank = predicted.rank(ascending=False, method="average")

        rows.append({
            "model": model,
            "true_best": true_best,
            "predicted_best": predicted_best,
            "top1_correct": int(predicted_best == true_best),
            "rank_correlation": true_rank.corr(predicted_rank),
        })

    return pd.DataFrame(rows)


display(
    evaluate_run(
        conn,
        frames["order_truth"],
    )
)


In [ ]:
def run_many(n_runs=30, n_users=N_USERS, first_seed=100):
    results = []

    for seed in range(first_seed, first_seed + n_runs):
        run_frames = generate_synthetic(
            seed=seed,
            n_users=n_users,
        )

        run_conn = write_db(
            run_frames,
            path=DB_PATH,
        )

        score = evaluate_run(
            run_conn,
            run_frames["order_truth"],
        )

        score["seed"] = seed
        results.append(score)
        run_conn.close()

    # Возвращаем demo seed в synthetic_experiment.db.
    final_frames = generate_synthetic(
        seed=SEED,
        n_users=n_users,
    )
    final_conn = write_db(
        final_frames,
        path=DB_PATH,
    )
    final_conn.close()

    return pd.concat(results, ignore_index=True)


runs = run_many(30)

result_summary = (
    runs.groupby("model")
    .agg(
        top1_accuracy=("top1_correct", "mean"),
        mean_rank_correlation=("rank_correlation", "mean"),
    )
    .sort_values("top1_accuracy", ascending=False)
)

display(result_summary)


In [ ]:
ax = (result_summary["top1_accuracy"] * 100).plot(kind="bar")
ax.set_ylabel("Top-1 accuracy, %")
ax.set_xlabel("Attribution model")
ax.set_title("Как часто модель находит placement с максимальным true ROMI")

plt.tight_layout()
plt.savefig(CHART_PATH, dpi=160)
plt.show()

print("chart:", CHART_PATH)


## Как читать результат

true_romi считается по скрытому true_driver. Модели этого поля не видят.

Атрибуция и ROMI считаются существующим src/core/attribution.py с окном 30 дней. От генератора нужны только synthetic-данные и ground truth для последующего сравнения результатов.
